# CSE476 Agentic AI and Intelligent Automation — Group 11
## E-Commerce Order Fulfillment Multi-Agent Orchestrator
**Group Members:** Ayush Dwivedi & Tushar  
**Framework:** Microsoft Agent Framework (`agent-framework-core` v1.26.0)  
**Architecture:** Sequential Pipeline with Fraud-Escalation Route  
**Memory:** Shared `OrderState` & Prior-Order Customer Historical Memory  
**Primary Deliverables:** Dispatched Order with Carrier Tracking ID, OR Escalated Hold with Reason


### 1. Architectural Pipeline & Specialised Agents
The fulfillment pipeline coordinates **five specialised agents** (plus a multi-channel **Notification Agent** extension):
1. **Validation Agent**: Customer standing, product existence, positive quantity bounds, address completeness, and payment authorization.
2. **Inventory Agent**: Real-time stock verification, atomic database reservations, shortage detection, substitute SKU proposals, and backorder proposals.
3. **Fraud Check Agent**: Deterministic 0-100 risk scoring evaluating prior dispute memory, retry velocity, order value anomalies, and shipping/billing mismatches.
4. **Pricing Agent**: Deterministic financial decimal arithmetic for subtotals, volume discounts (>=5 units), coupon codes, regional tax (8%), and dynamic shipping.
5. **Dispatch Coordinator**: Carrier selection (FedEx, UPS, USPS, DHL), unique tracking ID generation (`TRK-XXX-XXXX`), shipment persistence, and inventory commitment.
6. **Notification Agent**: Previews multi-channel customer communications across Confirmed, Substituted, and Held outcomes.

Every agent calls at least **TWO plain Python functions decorated with `@tool`** from Microsoft Agent Framework.


In [1]:
import os
import sys
import json
from decimal import Decimal

# Set workspace root
sys.path.insert(0, os.path.abspath("."))

from backend.app.database import init_db, SessionLocal
from backend.app.services.seed_data import seed_database
from backend.app.services.customer_memory import CustomerMemoryService
from backend.app.services.run_logger import AppendOnlyRunLogger
from backend.app.orchestrator.pipeline import OrderFulfillmentOrchestrator
from backend.app.schemas.order_state import OrderState, OrderItemState, AddressState, PaymentState
from backend.app.models.order import Order, OrderItem
from backend.app.models.shipment import Shipment
from backend.app.models.fraud import FraudReview
from backend.app.models.agent_log import AgentLog
from backend.app.models.inventory import InventoryReservation

print("[OK] Microsoft Agent Framework and Orchestrator modules successfully imported.")


[OK] Microsoft Agent Framework and Orchestrator modules successfully imported.


In [2]:
# Initialize SQLite database and seed initial test customers & catalog
init_db()
db = SessionLocal()
seed_database(db)

# Clean up previous demonstration orders for idempotent reruns
for tbl in [Shipment, FraudReview, AgentLog, InventoryReservation, OrderItem]:
    db.query(tbl).filter(tbl.order_id.like("ORD-NB-%")).delete()
db.query(Order).filter(Order.id.like("ORD-NB-%")).delete()
db.commit()

orchestrator = OrderFulfillmentOrchestrator()
print("[OK] Database initialized and clean demonstration workspace prepared.")


[OK] Database initialized and clean demonstration workspace prepared.


In [3]:
def display_execution_summary(order_state: OrderState):
    print("=" * 80)
    print(f"  ORDER EXECUTION SUMMARY: {order_state.order_id}")
    print("=" * 80)
    print(f"Customer ID:     {order_state.customer_id} ({order_state.customer_name})")
    print(f"Final Status:    {order_state.dispatch_status}")
    print(f"Validation:      {order_state.validation_status}")
    print(f"Inventory:       {order_state.inventory_status}")
    print(f"Fraud Status:    {order_state.fraud_status} (Score: {order_state.fraud_score}/100)")
    print(f"Pricing Status:  {order_state.pricing_status} (Final Amount: ${order_state.final_amount})")
    if order_state.tracking_id:
        print(f"Carrier Logistics: {order_state.carrier} ({order_state.shipping_method})")
        print(f"Tracking ID:     {order_state.tracking_id}")
    if order_state.hold_reason:
        print(f"Hold Reason:     {order_state.hold_reason}")
        
    print("")
    print("--- 5-AGENT PIPELINE EXECUTION TRACE ---")
    for log in order_state.agent_execution_logs:
        status_tag = "PASS" if log.status in ["SUCCESS", "PASSED"] else log.status
        print(f"  [{log.agent_name:<20}] {log.action:<26} -> {status_tag:<7} ({log.execution_time_ms:>2}ms): {log.message}")
        
    print("")
    print("--- MICROSOFT AGENT FRAMEWORK TOOL CALLS (Granular) ---")
    for tool_log in order_state.tool_execution_logs:
        print(f"  [@tool {tool_log.tool_name:<30}] -> {tool_log.status:<7} ({tool_log.execution_time_ms:>2}ms): {tool_log.message}")


### 2. Scenario 1: Clean Order with Successful Dispatch
A verified repeat customer (`CUST-001`) with complete address, valid payment token, and standard coupon `WELCOME10`.
- All 5 agents complete successfully.
- Final result: **Confirmed order + final amount + shipping carrier + tracking ID**.


In [4]:
state1 = OrderState(
    order_id="ORD-NB-01",
    customer_id="CUST-001",
    items=[OrderItemState(product_id="PROD-001", sku="KB-MECH-RGB", name="Pro Mechanical Keyboard", quantity=1)],
    shipping_address=AddressState(
        street="742 Evergreen Terrace",
        city="Springfield",
        state="OR",
        postal_code="97477",
        country="USA"
    ),
    payment_information=PaymentState(
        payment_method="CREDIT_CARD",
        payment_status="AUTHORIZED",
        transaction_token="tok_valid_live"
    ),
    coupon_code="WELCOME10",
)

res1 = orchestrator.process_order_pipeline(state1, db)
display_execution_summary(res1)


  ORDER EXECUTION SUMMARY: ORD-NB-01
Customer ID:     CUST-001 (John Verified)
Final Status:    DISPATCHED
Validation:      PASSED
Inventory:       RESERVED
Fraud Status:    LOW_RISK (Score: 0.0/100)
Pricing Status:  CALCULATED (Final Amount: $102.19)
Carrier Logistics: UPS Ground Logistics (Standard)
Tracking ID:     TRK-UPS-41881878

--- 5-AGENT PIPELINE EXECUTION TRACE ---
  [ValidationAgent     ] OrderValidationSummary     -> PASS    ( 2ms): Order validation passed successfully with 1 item(s).
  [InventoryAgent      ] InventoryReservationSummary -> PASS    (16ms): All items successfully reserved atomically for order ORD-NB-01.
  [FraudCheckAgent     ] FraudEvaluationSummary     -> PASS    ( 8ms): Fraud evaluation completed with score 0.0/100 (Tier: LOW). Decision: APPROVED.
  [PricingAgent        ] PricingSummary             -> PASS    ( 0ms): Pricing completed: Final Payable Amount is $102.19.
  [DispatchCoordinator ] OrderDispatchSummary       -> PASS    (20ms): Order ORD-NB-01 s

### 3. Scenario 2: Invalid Order (Structural & Payment Failure)
An incoming order containing a non-existent catalog SKU, blank address street, and `FAILED` payment status.
- Strict guardrail: Pipeline **halts immediately at Stage 1**.
- Inventory, Fraud, Pricing, and Dispatch agents are never invoked.


In [5]:
state2 = OrderState(
    order_id="ORD-NB-02",
    customer_id="CUST-001",
    items=[OrderItemState(product_id="PROD-UNKNOWN", sku="NON-EXISTENT-SKU", name="Bad Item", quantity=1)],
    shipping_address=AddressState(street="", city="Springfield", state="OR", postal_code="97477", country="USA"),
    payment_information=PaymentState(payment_method="CREDIT_CARD", payment_status="FAILED", transaction_token=""),
)

res2 = orchestrator.process_order_pipeline(state2, db)
display_execution_summary(res2)


  ORDER EXECUTION SUMMARY: ORD-NB-02
Customer ID:     CUST-001 (John Verified)
Final Status:    HELD
Validation:      FAILED
Inventory:       PENDING
Fraud Status:    PENDING (Score: 0.0/100)
Pricing Status:  PENDING (Final Amount: $0.00)
Hold Reason:     Validation Failed: Product with SKU 'NON-EXISTENT-SKU' does not exist in catalog.; Shipping address street is missing or empty.; Payment status is FAILED; transaction cannot proceed.; Payment authorization token is missing.

--- 5-AGENT PIPELINE EXECUTION TRACE ---
  [ValidationAgent     ] OrderValidationSummary     -> FAILED  ( 0ms): Order validation rejected with 4 error(s).

--- MICROSOFT AGENT FRAMEWORK TOOL CALLS (Granular) ---
  [@tool validate_customer_standing    ] -> SUCCESS ( 0ms): Customer standing check passed for CUST-001.
  [@tool validate_catalog_and_quantities] -> FAILED  ( 0ms): Catalog SKU check failed with 1 line item(s).
  [@tool validate_shipping_and_payment ] -> FAILED  ( 0ms): Address and payment authorization c

### 4. Scenario 3: High-Risk Fraudulent Order (Fraud Escalation Route)
Customer `CUST-003` has 2 past chargebacks, failed payment retry velocity, and shipping/billing mismatch.
- Fraud Check Agent scores order at **100/100 (>= 70 critical threshold)**.
- **Fraud Escalation Route**: Diverts order to Hold Queue for human auditor review.
- **Rollback Guarantee**: Atomic inventory reservations are released immediately.
- Order is blocked from entering Pricing or Dispatch.


In [6]:
state3 = OrderState(
    order_id="ORD-NB-03",
    customer_id="CUST-003",
    items=[OrderItemState(product_id="PROD-003", sku="MON-4K-27", name="4K Designer Monitor", quantity=3)],
    shipping_address=AddressState(street="PO Box 9999", city="Unknown Island", state="FL", postal_code="33040", country="USA"),
    payment_information=PaymentState(
        payment_method="CREDIT_CARD",
        payment_status="AUTHORIZED",
        transaction_token="tok_suspicious",
        billing_address=AddressState(street="100 Glacier Way", city="Anchorage", state="AK", postal_code="99501", country="USA")
    ),
)

res3 = orchestrator.process_order_pipeline(state3, db, simulated_failed_attempts=2)
display_execution_summary(res3)


  ORDER EXECUTION SUMMARY: ORD-NB-03
Customer ID:     CUST-003 (Alex Risky)
Final Status:    HELD
Validation:      PASSED
Inventory:       RESERVED
Fraud Status:    HIGH_RISK_HELD (Score: 100.0/100)
Pricing Status:  PENDING (Final Amount: $0.00)
Hold Reason:     Fraud Escalation Hold: Risk score of 100.0/100 exceeds critical threshold (70). Triggers: Customer has 2 prior chargebacks / suspicious orders (+45); High payment retry velocity (7 failed attempts) (+30); Shipping (Unknown Island, FL) and Billing (Anchorage, AK) mismatch (+15); Destination matches PO Box / Mail Drop profile (+10)

--- 5-AGENT PIPELINE EXECUTION TRACE ---
  [ValidationAgent     ] OrderValidationSummary     -> PASS    ( 0ms): Order validation passed successfully with 1 item(s).
  [InventoryAgent      ] InventoryReservationSummary -> PASS    (19ms): All items successfully reserved atomically for order ORD-NB-03.
  [FraudCheckAgent     ] FraudEvaluationSummary     -> HELD    (14ms): Fraud evaluation completed with 

### 5. Scenario 4: Insufficient Inventory (No Silent Substitution)
Order requests an out-of-stock ultrawide monitor (`MON-CRV-32`).
- Inventory Agent detects stock shortage and identifies compatible substitute (`MON-4K-27`) and backorder timeline (7 days).
- **Strict Business Rule**: Never silently alter an order. Customer approval required before changing order.
- Order is held with clear proposed substitute.


In [7]:
state4 = OrderState(
    order_id="ORD-NB-04",
    customer_id="CUST-001",
    items=[OrderItemState(product_id="PROD-004", sku="MON-CRV-32", name="Curved Ultrawide Monitor", quantity=1)],
    shipping_address=AddressState(street="742 Evergreen Terrace", city="Springfield", state="OR", postal_code="97477", country="USA"),
    payment_information=PaymentState(payment_method="CREDIT_CARD", payment_status="AUTHORIZED", transaction_token="tok_valid_d4"),
)

res4 = orchestrator.process_order_pipeline(state4, db)
display_execution_summary(res4)


  ORDER EXECUTION SUMMARY: ORD-NB-04
Customer ID:     CUST-001 (John Verified)
Final Status:    HELD
Validation:      PASSED
Inventory:       SUBSTITUTE_RECOMMENDED
Fraud Status:    PENDING (Score: 0.0/100)
Pricing Status:  PENDING (Final Amount: $0.00)
Hold Reason:     Inventory Notice: Out of stock. Substitute suggested: Ultra HD 27-inch 4K Designer Monitor (MON-4K-27). Customer approval required before changing order.

--- 5-AGENT PIPELINE EXECUTION TRACE ---
  [ValidationAgent     ] OrderValidationSummary     -> PASS    ( 5ms): Order validation passed successfully with 1 item(s).
  [InventoryAgent      ] InventoryHoldDecision      -> HELD    ( 0ms): Inventory Notice: Out of stock. Substitute suggested: Ultra HD 27-inch 4K Designer Monitor (MON-4K-27). Customer approval required before changing order.

--- MICROSOFT AGENT FRAMEWORK TOOL CALLS (Granular) ---
  [@tool validate_customer_standing    ] -> SUCCESS ( 5ms): Customer standing check passed for CUST-001.
  [@tool validate_cata

### 6. Scenario 5: Customer Historical Memory Inspection
Prior-order memory retrieved by `CustomerMemoryService` and used by `FraudCheckAgent`.


In [8]:
print("--- CUSTOMER 1 (TRUSTED REPEAT PROFILE) ---")
h1 = CustomerMemoryService.get_customer_history(db, "CUST-001")
print(f"Total Orders:      {h1.total_orders}")
print(f"Successful Orders: {h1.successful_orders}")
print(f"Suspicious Orders: {h1.suspicious_orders}")
print(f"Average Spend:     ${h1.average_order_value:.2f}")
print(f"Risk Tier:         {h1.risk_tier}")

print("")
print("--- CUSTOMER 3 (HIGH RISK / DISPUTE PROFILE) ---")
h3 = CustomerMemoryService.get_customer_history(db, "CUST-003")
print(f"Total Orders:      {h3.total_orders}")
print(f"Successful Orders: {h3.successful_orders}")
print(f"Suspicious Orders: {h3.suspicious_orders}")
print(f"Suspicion Rate:    {h3.suspicion_rate * 100:.1f}%")
print(f"Risk Tier:         {h3.risk_tier}")


--- CUSTOMER 1 (TRUSTED REPEAT PROFILE) ---
Total Orders:      12
Successful Orders: 12
Suspicious Orders: 0
Average Spend:     $159.82
Risk Tier:         LOW

--- CUSTOMER 3 (HIGH RISK / DISPUTE PROFILE) ---
Total Orders:      3
Successful Orders: 1
Suspicious Orders: 2
Suspicion Rate:    66.7%
Risk Tier:         HIGH


### 7. Customer Notification Draft Preview (Group Extension)
The `NotificationAgent` drafts personalized customer communications across Email and SMS channels without transmitting real messages externally without credentials.


In [9]:
print("=== DRAFTED NOTIFICATION FOR CONFIRMED ORDER (ORD-NB-01) ===")
draft1 = res1.customer_notification_draft
print(f"Channel: {draft1.channel}")
print(f"Subject: {draft1.subject}")
print(f"Body:\n{draft1.body}")
print(f"SMS:     {draft1.sms_summary}")

print("")
print("=== DRAFTED NOTIFICATION FOR HELD FRAUD ORDER (ORD-NB-03) ===")
draft3 = res3.customer_notification_draft
print(f"Subject: {draft3.subject}")
print(f"Body:\n{draft3.body}")


=== DRAFTED NOTIFICATION FOR CONFIRMED ORDER (ORD-NB-01) ===
Channel: EMAIL_AND_SMS
Subject: Order Confirmed & Shipped! (ORD-NB-01)
Body:
Dear John Verified,

Great news! Your order ORD-NB-01 has been confirmed, carefully packaged, and dispatched.

Carrier: UPS Ground Logistics (Standard)
Tracking ID: TRK-UPS-41881878
Total Paid: $102.19

You can track your delivery in real time through our operations dashboard.

Thank you for shopping with us!
SMS:     Your order ORD-NB-01 has shipped via UPS Ground Logistics! Tracking: TRK-UPS-41881878.

=== DRAFTED NOTIFICATION FOR HELD FRAUD ORDER (ORD-NB-03) ===
Subject: Security Verification: Order ORD-NB-03 Under Review
Body:
Dear Alex Risky,

Thank you for your order ORD-NB-03. As part of our standard security procedures to protect our cardholders, this order has been routed to our verification queue for routine confirmation.

No additional charges have been applied. Our customer trust and safety team will complete this review within 1 business

### 8. Append-Only Run Log Verification
Every pipeline execution is committed to `logs/execution_run_log.jsonl` sequentially without overwriting past entries.


In [10]:
recent_runs = AppendOnlyRunLogger.read_recent_runs(limit=3)
print(f"Retrieved {len(recent_runs)} recent append-only execution records:\n")
for run in recent_runs:
    print(f"Run ID:      {run['run_id']}")
    print(f"Timestamp:   {run['timestamp']}")
    print(f"Order ID:    {run['order_id']} -> {run['dispatch_status']}")
    print(f"Total Steps: {run['total_steps']} agent logs, {run['total_tool_calls']} tool calls")
    print("-" * 60)


Retrieved 3 recent append-only execution records:

Run ID:      RUN-4363B37F9068
Timestamp:   2026-11-22T18:30:03.535231+00:00
Order ID:    ORD-NB-04 -> HELD
Total Steps: 2 agent logs, 7 tool calls
------------------------------------------------------------
Run ID:      RUN-91D7FBBFF43D
Timestamp:   2026-11-22T18:30:03.487783+00:00
Order ID:    ORD-NB-03 -> HELD
Total Steps: 3 agent logs, 11 tool calls
------------------------------------------------------------
Run ID:      RUN-534DDAE707C6
Timestamp:   2026-11-22T18:30:03.398121+00:00
Order ID:    ORD-NB-02 -> HELD
Total Steps: 1 agent logs, 6 tool calls
------------------------------------------------------------


## 9. Viva Defense Quick Reference
- **Why Pipeline Architecture?** E-commerce fulfillment has strict operational prerequisites: an order cannot reserve inventory if invalid; cannot calculate pricing or dispatch if fraudulent; and must release reservations on cancellation.
- **Why Microsoft Agent Framework?** Provides standard `BaseAgent` and `@tool` abstractions, tool invocation metadata, schema validation, and modular decoupling.
- **How is Atomic Stock Integrity Guaranteed?** Transactions lock rows (`with_for_update`) during verification; if an order is diverted to the fraud hold queue or cancelled, reservations are released back to stock.
- **How Does Customer Memory Prevent Fraud?** Repeat customers with clean delivery histories gain low baseline scores; users with past chargebacks or rapid payment retries are flagged or escalated.
